# Rossmann Retail Sales Forecasting & FP&A Analytics
## 01 — Data Audit

### Purpose

This notebook validates the four original Kaggle files before any cleaning, feature engineering, or modelling is performed.

### Audit scope

- File availability and size
- Dataset dimensions and columns
- Inferred data types
- Date validity and coverage
- Missing values
- Exact and key-level duplicates
- Dataset granularity
- Train/test schema differences
- Basic business-rule checks
- Referential integrity between daily records and the store master

> **Stop condition:** do not clean or impute data in this notebook. Review and document the audit results first.

## 1. Business framing and initial decisions

- **Business objective:** forecast daily sales by store to support FP&A, performance management, and operational planning.
- **Target:** `Sales`.
- **Expected prediction grain:** one row per `Store` and `Date`.
- **Forecast horizon:** to be confirmed after the date audit.
- **`Customers`:** available historically but absent from the Kaggle test set. It may be used descriptively, but not as a principal production predictor.
- **Validation:** chronological validation will be used later; a conventional random split is not appropriate.
- **Kaggle test data:** reserved for final prediction generation because it does not contain observed sales.

In [1]:
from pathlib import Path
import platform

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

print(f"Python: {platform.python_version()}")
print(f"pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

Python: 3.11.15
pandas: 2.3.3
NumPy: 2.4.6


## 2. Locate the project and validate expected files

The code searches the current directory and its parents for the repository structure. This allows the notebook to work whether Jupyter starts in the project root or in the `notebooks/` folder.

In [2]:
def find_project_root(start_path: Path) -> Path:
    """Return the nearest parent containing the expected project folders."""
    start_path = start_path.resolve()
    candidates = [start_path, *start_path.parents]

    for candidate in candidates:
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate

    raise FileNotFoundError(
        "Project root not found. Open Jupyter from the repository folder "
        "or confirm that the data/ and notebooks/ folders exist."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
REPORT_TABLES_DIR = PROJECT_ROOT / "reports" / "tables"

EXPECTED_FILES = {
    "train": RAW_DATA_DIR / "train.csv",
    "store": RAW_DATA_DIR / "store.csv",
    "test": RAW_DATA_DIR / "test.csv",
    "sample_submission": RAW_DATA_DIR / "sample_submission.csv",
}

missing_files = [
    str(path.relative_to(PROJECT_ROOT))
    for path in EXPECTED_FILES.values()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Missing expected files:\n- " + "\n- ".join(missing_files)
    )

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data folder: {RAW_DATA_DIR}")
print("All expected files are available.")

Project root: C:\Users\koldo\Desktop\Máster de DATA Science con IA\PROYECTO\rossmann-sales-forecasting
Raw data folder: C:\Users\koldo\Desktop\Máster de DATA Science con IA\PROYECTO\rossmann-sales-forecasting\data\raw
All expected files are available.


## 3. File inventory

File size is checked before loading. Unexpected sizes can indicate an incomplete download, a wrong file, or accidental modification.

In [3]:
file_inventory = pd.DataFrame(
    [
        {
            "dataset": dataset_name,
            "file_name": file_path.name,
            "size_bytes": file_path.stat().st_size,
            "size_kb": file_path.stat().st_size / 1_024,
            "size_mb": file_path.stat().st_size / (1_024 ** 2),
        }
        for dataset_name, file_path in EXPECTED_FILES.items()
    ]
).sort_values("size_bytes", ascending=False)

display(file_inventory)

,dataset,file_name,size_bytes,size_kb,size_mb
0,train,train.csv,38057952,"37,165.97",36.29
2,test,test.csv,1427425,"1,393.97",1.36
3,sample_submission,sample_submission.csv,317611,310.17,0.30
1,store,store.csv,45010,43.96,0.04


## 4. Load the raw CSV files

The files are loaded without transformations. `low_memory=False` asks pandas to infer each column using the full file rather than independent chunks.

In [5]:
datasets = {
    dataset_name: pd.read_csv(file_path, low_memory=False)
    for dataset_name, file_path in EXPECTED_FILES.items()
}

train = datasets["train"]
store = datasets["store"]
test = datasets["test"]
sample_submission = datasets["sample_submission"]

print("Raw files loaded successfully.")

Raw files loaded successfully.


## 5. Dataset dimensions and memory

This establishes the physical size of each table and provides the first check that the downloaded files are coherent.

In [6]:
dataset_overview = pd.DataFrame(
    [
        {
            "dataset": dataset_name,
            "rows": dataframe.shape[0],
            "columns": dataframe.shape[1],
            "memory_mb": dataframe.memory_usage(deep=True).sum() / (1_024 ** 2),
        }
        for dataset_name, dataframe in datasets.items()
    ]
).sort_values("rows", ascending=False)

display(dataset_overview)

,dataset,rows,columns,memory_mb
0,train,1017209,9,175.59
2,test,41088,8,6.78
3,sample_submission,41088,2,0.63
1,store,1115,10,0.24


## 6. Column names and record samples

The first and last rows help identify ordering, formatting, and obvious structural issues without changing the data.

In [7]:
for dataset_name, dataframe in datasets.items():
    print(f"\n{'=' * 80}")
    print(f"{dataset_name.upper()} — COLUMNS")
    print(dataframe.columns.tolist())

    print(f"\n{dataset_name.upper()} — FIRST 3 ROWS")
    display(dataframe.head(3))

    print(f"{dataset_name.upper()} — LAST 3 ROWS")
    display(dataframe.tail(3))


TRAIN — COLUMNS
['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']

TRAIN — FIRST 3 ROWS


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1


TRAIN — LAST 3 ROWS


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
1017206,1113,2,2013-01-01,0,0,0,0,a,1
1017207,1114,2,2013-01-01,0,0,0,0,a,1
1017208,1115,2,2013-01-01,0,0,0,0,a,1



STORE — COLUMNS
['Store', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval']

STORE — FIRST 3 ROWS


...

,Id,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday
41085,41086,1113,6,2015-08-01,1.00,0,0,0
41086,41087,1114,6,2015-08-01,1.00,0,0,0
41087,41088,1115,6,2015-08-01,1.00,0,0,1



SAMPLE_SUBMISSION — COLUMNS
['Id', 'Sales']

SAMPLE_SUBMISSION — FIRST 3 ROWS


,Id,Sales
0,1,0
1,2,0
2,3,0


SAMPLE_SUBMISSION — LAST 3 ROWS


,Id,Sales
41085,41086,0
41086,41087,0
41087,41088,0


## 7. Inferred data types

A pandas type is only an initial inference. For example, identifiers may be numeric without being continuous measures, and a column containing mixed codes may be loaded as text.

In [11]:
dtype_tables = []

for dataset_name, dataframe in datasets.items():
    dtype_table = (
        dataframe.dtypes.astype(str)
        .rename_axis("column")
        .reset_index(name="dtype")
    )
    dtype_table.insert(0, "dataset", dataset_name)
    dtype_tables.append(dtype_table)

dtype_summary = pd.concat(dtype_tables, ignore_index=True)
display(dtype_summary)

,dataset,column,dtype
0,train,Store,int64
1,train,DayOfWeek,int64
2,train,Date,object
3,train,Sales,int64
4,train,Customers,int64
5,train,Open,int64
6,train,Promo,int64
7,train,StateHoliday,object
8,train,SchoolHoliday,int64
9,store,Store,int64


## 8. Missing values

Both the count and percentage are reported. Missingness will be interpreted later because some nulls may be structural rather than data-quality errors.

In [12]:
missing_tables = []

for dataset_name, dataframe in datasets.items():
    missing_table = pd.DataFrame(
        {
            "column": dataframe.columns,
            "missing_count": dataframe.isna().sum().values,
            "missing_pct": (dataframe.isna().mean().values * 100),
        }
    )
    missing_table.insert(0, "dataset", dataset_name)
    missing_tables.append(missing_table)

missing_summary = pd.concat(missing_tables, ignore_index=True)
missing_summary = missing_summary.sort_values(
    ["missing_pct", "missing_count"],
    ascending=False,
)

display(missing_summary[missing_summary["missing_count"] > 0])

if missing_summary["missing_count"].sum() == 0:
    print("No missing values detected in any dataset.")

,dataset,column,missing_count,missing_pct
16,store,Promo2SinceWeek,544,48.79
17,store,Promo2SinceYear,544,48.79
18,store,PromoInterval,544,48.79
13,store,CompetitionOpenSinceMonth,354,31.75
14,store,CompetitionOpenSinceYear,354,31.75
12,store,CompetitionDistance,3,0.27
23,test,Open,11,0.03


## 9. Exact duplicate rows

Exact duplicates are checked separately from duplicate business keys. Two records can differ in one field but still violate the expected one-row-per-store-date grain.

In [13]:
exact_duplicate_summary = pd.DataFrame(
    [
        {
            "dataset": dataset_name,
            "exact_duplicate_rows": int(dataframe.duplicated().sum()),
        }
        for dataset_name, dataframe in datasets.items()
    ]
)

display(exact_duplicate_summary)

,dataset,exact_duplicate_rows
0,train,0
1,store,0
2,test,0
3,sample_submission,0


## 10. Granularity and key uniqueness

Expected keys:

- `train`: `Store` + `Date`
- `test`: `Store` + `Date`
- `store`: `Store`
- `test`: `Id`
- `sample_submission`: `Id`

In [14]:
key_checks = []

def add_key_check(
    dataset_name: str,
    dataframe: pd.DataFrame,
    key_columns: list[str],
) -> None:
    missing_key_columns = [
        column for column in key_columns if column not in dataframe.columns
    ]

    if missing_key_columns:
        key_checks.append(
            {
                "dataset": dataset_name,
                "key": " + ".join(key_columns),
                "duplicate_key_rows": np.nan,
                "unique_key": False,
                "note": f"Missing columns: {missing_key_columns}",
            }
        )
        return

    duplicate_key_rows = int(
        dataframe.duplicated(subset=key_columns, keep=False).sum()
    )
    key_checks.append(
        {
            "dataset": dataset_name,
            "key": " + ".join(key_columns),
            "duplicate_key_rows": duplicate_key_rows,
            "unique_key": duplicate_key_rows == 0,
            "note": "",
        }
    )


add_key_check("train", train, ["Store", "Date"])
add_key_check("test", test, ["Store", "Date"])
add_key_check("store", store, ["Store"])
add_key_check("test", test, ["Id"])
add_key_check("sample_submission", sample_submission, ["Id"])

key_summary = pd.DataFrame(key_checks)
display(key_summary)

,dataset,key,duplicate_key_rows,unique_key,note
0,train,Store + Date,0,True,
1,test,Store + Date,0,True,
2,store,Store,0,True,
3,test,Id,0,True,
4,sample_submission,Id,0,True,


## 11. Date validity, coverage, and weekday consistency

Dates are parsed into temporary Series. The original DataFrames remain unchanged.

Pandas uses Monday = 0, while the Rossmann `DayOfWeek` convention is expected to use Monday = 1. Adding 1 makes the values comparable.

In [15]:
date_audit_rows = []
parsed_dates = {}

for dataset_name in ["train", "test"]:
    dataframe = datasets[dataset_name]

    if "Date" not in dataframe.columns:
        date_audit_rows.append(
            {
                "dataset": dataset_name,
                "invalid_dates": np.nan,
                "min_date": pd.NaT,
                "max_date": pd.NaT,
                "unique_dates": np.nan,
                "calendar_days_in_range": np.nan,
                "missing_calendar_dates": np.nan,
                "weekday_mismatches": np.nan,
            }
        )
        continue

    parsed_date = pd.to_datetime(dataframe["Date"], errors="coerce")
    parsed_dates[dataset_name] = parsed_date

    valid_dates = parsed_date.dropna()
    min_date = valid_dates.min()
    max_date = valid_dates.max()
    calendar_days_in_range = (
        (max_date - min_date).days + 1 if not valid_dates.empty else np.nan
    )
    unique_dates = valid_dates.nunique()
    missing_calendar_dates = (
        calendar_days_in_range - unique_dates
        if pd.notna(calendar_days_in_range)
        else np.nan
    )

    weekday_mismatches = np.nan
    if "DayOfWeek" in dataframe.columns:
        expected_day_of_week = parsed_date.dt.dayofweek + 1
        weekday_mismatches = int(
            (
                parsed_date.notna()
                & dataframe["DayOfWeek"].notna()
                & (dataframe["DayOfWeek"] != expected_day_of_week)
            ).sum()
        )

    date_audit_rows.append(
        {
            "dataset": dataset_name,
            "invalid_dates": int(parsed_date.isna().sum()),
            "min_date": min_date,
            "max_date": max_date,
            "unique_dates": unique_dates,
            "calendar_days_in_range": calendar_days_in_range,
            "missing_calendar_dates": missing_calendar_dates,
            "weekday_mismatches": weekday_mismatches,
        }
    )

date_audit = pd.DataFrame(date_audit_rows)
display(date_audit)

for dataset_name, parsed_date in parsed_dates.items():
    coverage_by_store = (
        datasets[dataset_name]
        .assign(_parsed_date=parsed_date)
        .groupby("Store", as_index=False)
        .agg(
            records=("Store", "size"),
            min_date=("_parsed_date", "min"),
            max_date=("_parsed_date", "max"),
            unique_dates=("_parsed_date", "nunique"),
        )
    )

    print(f"\n{dataset_name.upper()} — STORE-LEVEL DATE COVERAGE")
    display(coverage_by_store.describe(include="all").transpose())

,dataset,invalid_dates,min_date,max_date,unique_dates,calendar_days_in_range,missing_calendar_dates,weekday_mismatches
0,train,0,2013-01-01,2015-07-31,942,942,0,0
1,test,0,2015-08-01,2015-09-17,48,48,0,0



TRAIN — STORE-LEVEL DATE COVERAGE


,count,mean,min,25%,50%,75%,max,std
Store,"1,115.00",558.00,1.00,279.50,558.00,836.50,"1,115.00",322.02
records,"1,115.00",912.30,758.00,942.00,942.00,942.00,942.00,67.73
min_date,1115,2013-01-01 00:01:17.488789248,2013-01-01 00:00:00,2013-01-01 00:00:00,2013-01-01 00:00:00,2013-01-01 00:00:00,2013-01-02 00:00:00,NaN
max_date,1115,2015-07-31 00:00:00,2015-07-31 00:00:00,2015-07-31 00:00:00,2015-07-31 00:00:00,2015-07-31 00:00:00,2015-07-31 00:00:00,NaN
unique_dates,"1,115.00",912.30,758.00,942.00,942.00,942.00,942.00,67.73



TEST — STORE-LEVEL DATE COVERAGE


,count,mean,min,25%,50%,75%,max,std
Store,856.00,555.90,1.00,279.75,553.50,832.25,"1,115.00",320.46
records,856.00,48.00,48.00,48.00,48.00,48.00,48.00,0.00
min_date,856,2015-08-01 00:00:00,2015-08-01 00:00:00,2015-08-01 00:00:00,2015-08-01 00:00:00,2015-08-01 00:00:00,2015-08-01 00:00:00,NaN
max_date,856,2015-09-17 00:00:00,2015-09-17 00:00:00,2015-09-17 00:00:00,2015-09-17 00:00:00,2015-09-17 00:00:00,2015-09-17 00:00:00,NaN
unique_dates,856.00,48.00,48.00,48.00,48.00,48.00,48.00,0.00


## 12. Train/test schema comparison

This identifies columns unavailable at prediction time and common columns whose inferred types differ.

In [16]:
train_columns = set(train.columns)
test_columns = set(test.columns)

schema_comparison = {
    "only_in_train": sorted(train_columns - test_columns),
    "only_in_test": sorted(test_columns - train_columns),
    "common_columns": sorted(train_columns & test_columns),
}

print("Only in train:", schema_comparison["only_in_train"])
print("Only in test:", schema_comparison["only_in_test"])
print("Common columns:", schema_comparison["common_columns"])

common_dtype_comparison = pd.DataFrame(
    {
        "column": schema_comparison["common_columns"],
        "train_dtype": [
            str(train[column].dtype)
            for column in schema_comparison["common_columns"]
        ],
        "test_dtype": [
            str(test[column].dtype)
            for column in schema_comparison["common_columns"]
        ],
    }
)
common_dtype_comparison["same_inferred_dtype"] = (
    common_dtype_comparison["train_dtype"]
    == common_dtype_comparison["test_dtype"]
)

display(common_dtype_comparison)

Only in train: ['Customers', 'Sales']
Only in test: ['Id']
Common columns: ['Date', 'DayOfWeek', 'Open', 'Promo', 'SchoolHoliday', 'StateHoliday', 'Store']


,column,train_dtype,test_dtype,same_inferred_dtype
0,Date,object,object,True
1,DayOfWeek,int64,int64,True
2,Open,int64,float64,False
3,Promo,int64,int64,True
4,SchoolHoliday,int64,int64,True
5,StateHoliday,object,object,True
6,Store,int64,int64,True


## 13. Value domains for selected categorical and indicator columns

Frequency tables help detect unexpected codes, inconsistent labels, and rare categories.

In [17]:
domain_columns = {
    "train": [
        "DayOfWeek",
        "Open",
        "Promo",
        "StateHoliday",
        "SchoolHoliday",
    ],
    "test": [
        "DayOfWeek",
        "Open",
        "Promo",
        "StateHoliday",
        "SchoolHoliday",
    ],
    "store": [
        "StoreType",
        "Assortment",
        "Promo2",
        "PromoInterval",
    ],
}

for dataset_name, columns in domain_columns.items():
    dataframe = datasets[dataset_name]
    print(f"\n{'=' * 80}")
    print(f"{dataset_name.upper()} — VALUE DOMAINS")

    for column in columns:
        if column not in dataframe.columns:
            continue

        counts = (
            dataframe[column]
            .value_counts(dropna=False)
            .rename_axis(column)
            .reset_index(name="count")
        )
        counts["pct"] = counts["count"] / len(dataframe) * 100

        print(f"\nColumn: {column}")
        display(counts)


TRAIN — VALUE DOMAINS

Column: DayOfWeek


,DayOfWeek,count,pct
0,5,145845,14.34
1,4,145845,14.34
2,3,145665,14.32
3,2,145664,14.32
4,1,144730,14.23
5,7,144730,14.23
6,6,144730,14.23



Column: Open


,Open,count,pct
0,1,844392,83.01
1,0,172817,16.99



Column: Promo


...

,Assortment,count,pct
0,a,593,53.18
1,c,513,46.01
2,b,9,0.81



Column: Promo2


,Promo2,count,pct
0,1,571,51.21
1,0,544,48.79



Column: PromoInterval


,PromoInterval,count,pct
0,NaN,544,48.79
1,"Jan,Apr,Jul,Oct",335,30.04
2,"Feb,May,Aug,Nov",130,11.66
3,"Mar,Jun,Sept,Dec",106,9.51


## 14. Basic business-rule checks

These tests flag potentially important cases. A flag is not automatically an error; the business meaning must be interpreted before any treatment is applied.

In [18]:
business_rule_checks = []

def add_rule(dataset: str, rule: str, affected_rows: int, note: str = "") -> None:
    business_rule_checks.append(
        {
            "dataset": dataset,
            "rule": rule,
            "affected_rows": int(affected_rows),
            "note": note,
        }
    )


if "Sales" in train.columns:
    add_rule("train", "Sales < 0", (train["Sales"] < 0).sum())
    add_rule("train", "Sales == 0", (train["Sales"] == 0).sum())

if "Customers" in train.columns:
    add_rule("train", "Customers < 0", (train["Customers"] < 0).sum())
    add_rule("train", "Customers == 0", (train["Customers"] == 0).sum())

if {"Open", "Sales"}.issubset(train.columns):
    add_rule(
        "train",
        "Open == 0 and Sales != 0",
        ((train["Open"] == 0) & (train["Sales"] != 0)).sum(),
    )
    add_rule(
        "train",
        "Open == 1 and Sales == 0",
        ((train["Open"] == 1) & (train["Sales"] == 0)).sum(),
        "Requires interpretation; may represent exceptional closures or data issues.",
    )

for dataset_name in ["train", "test"]:
    dataframe = datasets[dataset_name]

    if "Open" in dataframe.columns:
        add_rule(
            dataset_name,
            "Open outside {0, 1}, excluding missing",
            (
                dataframe["Open"].notna()
                & ~dataframe["Open"].isin([0, 1])
            ).sum(),
        )

    if "Promo" in dataframe.columns:
        add_rule(
            dataset_name,
            "Promo outside {0, 1}, excluding missing",
            (
                dataframe["Promo"].notna()
                & ~dataframe["Promo"].isin([0, 1])
            ).sum(),
        )

business_rule_summary = pd.DataFrame(business_rule_checks)
display(business_rule_summary)

,dataset,rule,affected_rows,note
0,train,Sales < 0,0,
1,train,Sales == 0,172871,
2,train,Customers < 0,0,
3,train,Customers == 0,172869,
4,train,Open == 0 and Sales != 0,0,
5,train,Open == 1 and Sales == 0,54,Requires interpretation; may represent excepti...
6,train,"Open outside {0, 1}, excluding missing",0,
7,train,"Promo outside {0, 1}, excluding missing",0,
8,test,"Open outside {0, 1}, excluding missing",0,
9,test,"Promo outside {0, 1}, excluding missing",0,


## 15. Referential integrity between daily records and the store master

Every store appearing in train or test should have one corresponding record in `store.csv`.

In [19]:
store_master_ids = set(store["Store"].dropna().unique())

referential_integrity = []

for dataset_name in ["train", "test"]:
    dataframe = datasets[dataset_name]
    dataset_store_ids = set(dataframe["Store"].dropna().unique())

    referential_integrity.append(
        {
            "dataset": dataset_name,
            "unique_stores": len(dataset_store_ids),
            "stores_missing_from_store_master": len(
                dataset_store_ids - store_master_ids
            ),
            "store_master_rows_not_used": len(
                store_master_ids - dataset_store_ids
            ),
        }
    )

referential_integrity_summary = pd.DataFrame(referential_integrity)
display(referential_integrity_summary)

,dataset,unique_stores,stores_missing_from_store_master,store_master_rows_not_used
0,train,1115,0,0
1,test,856,0,259


## 16. Test/submission alignment

The submission template should contain exactly one row for every test `Id`.

In [20]:
submission_alignment = pd.DataFrame(
    [
        {
            "check": "Same number of rows",
            "result": len(test) == len(sample_submission),
            "test_value": len(test),
            "submission_value": len(sample_submission),
        },
        {
            "check": "Same set of Id values",
            "result": set(test["Id"]) == set(sample_submission["Id"]),
            "test_value": test["Id"].nunique(),
            "submission_value": sample_submission["Id"].nunique(),
        },
        {
            "check": "Same Id order",
            "result": test["Id"].reset_index(drop=True).equals(
                sample_submission["Id"].reset_index(drop=True)
            ),
            "test_value": "Compared row by row",
            "submission_value": "Compared row by row",
        },
    ]
)

display(submission_alignment)

,check,result,test_value,submission_value
0,Same number of rows,True,41088,41088
1,Same set of Id values,True,41088,41088
2,Same Id order,True,Compared row by row,Compared row by row


## 17. Initial structural data dictionary

This is an automated structural dictionary. Business definitions and modelling treatment will be added later in `docs/data_dictionary.md`.

In [21]:
data_dictionary_rows = []

for dataset_name, dataframe in datasets.items():
    for column in dataframe.columns:
        data_dictionary_rows.append(
            {
                "dataset": dataset_name,
                "column": column,
                "dtype": str(dataframe[column].dtype),
                "non_null_count": int(dataframe[column].notna().sum()),
                "missing_count": int(dataframe[column].isna().sum()),
                "missing_pct": float(dataframe[column].isna().mean() * 100),
                "unique_values": int(dataframe[column].nunique(dropna=True)),
            }
        )

data_dictionary_initial = pd.DataFrame(data_dictionary_rows)
display(data_dictionary_initial)

,dataset,column,dtype,non_null_count,missing_count,missing_pct,unique_values
0,train,Store,int64,1017209,0,0.00,1115
1,train,DayOfWeek,int64,1017209,0,0.00,7
2,train,Date,object,1017209,0,0.00,942
3,train,Sales,int64,1017209,0,0.00,21734
4,train,Customers,int64,1017209,0,0.00,4086
5,train,Open,int64,1017209,0,0.00,2
6,train,Promo,int64,1017209,0,0.00,2
7,train,StateHoliday,object,1017209,0,0.00,4
8,train,SchoolHoliday,int64,1017209,0,0.00,2
9,store,Store,int64,1115,0,0.00,1115


## 18. Consolidated audit controls

The following table creates a concise checklist for the project documentation. It reports observations rather than applying fixes.

In [22]:
audit_controls = []

def add_control(area: str, check: str, passed: bool, observed_value) -> None:
    audit_controls.append(
        {
            "area": area,
            "check": check,
            "passed": bool(passed),
            "observed_value": observed_value,
        }
    )


for dataset_name, dataframe in datasets.items():
    add_control(
        "Files",
        f"{dataset_name}: non-empty dataset",
        len(dataframe) > 0 and dataframe.shape[1] > 0,
        f"{dataframe.shape[0]:,} rows x {dataframe.shape[1]} columns",
    )
    add_control(
        "Duplicates",
        f"{dataset_name}: no exact duplicate rows",
        dataframe.duplicated().sum() == 0,
        int(dataframe.duplicated().sum()),
    )

for _, row in key_summary.iterrows():
    add_control(
        "Granularity",
        f"{row['dataset']}: unique key {row['key']}",
        bool(row["unique_key"]),
        row["duplicate_key_rows"],
    )

for _, row in date_audit.iterrows():
    add_control(
        "Dates",
        f"{row['dataset']}: no invalid Date values",
        row["invalid_dates"] == 0,
        row["invalid_dates"],
    )
    add_control(
        "Dates",
        f"{row['dataset']}: DayOfWeek matches Date",
        row["weekday_mismatches"] == 0,
        row["weekday_mismatches"],
    )

for _, row in referential_integrity_summary.iterrows():
    add_control(
        "Referential integrity",
        f"{row['dataset']}: every Store exists in store master",
        row["stores_missing_from_store_master"] == 0,
        row["stores_missing_from_store_master"],
    )

add_control(
    "Submission",
    "test and sample_submission contain the same Id values",
    set(test["Id"]) == set(sample_submission["Id"]),
    f"test={test['Id'].nunique():,}; submission={sample_submission['Id'].nunique():,}",
)

audit_control_summary = pd.DataFrame(audit_controls)
display(audit_control_summary)

print("\nControls requiring review:")
display(audit_control_summary[~audit_control_summary["passed"]])

,area,check,passed,observed_value
0,Files,train: non-empty dataset,True,"1,017,209 rows x 9 columns"
1,Duplicates,train: no exact duplicate rows,True,0
2,Files,store: non-empty dataset,True,"1,115 rows x 10 columns"
3,Duplicates,store: no exact duplicate rows,True,0
4,Files,test: non-empty dataset,True,"41,088 rows x 8 columns"
5,Duplicates,test: no exact duplicate rows,True,0
6,Files,sample_submission: non-empty dataset,True,"41,088 rows x 2 columns"
7,Duplicates,sample_submission: no exact duplicate rows,True,0
8,Granularity,train: unique key Store + Date,True,0
9,Granularity,test: unique key Store + Date,True,0



Controls requiring review:


,area,check,passed,observed_value


## 19. Save generated audit tables

Only generated summaries are saved. The raw CSV files remain unchanged.

In [23]:
REPORT_TABLES_DIR.mkdir(parents=True, exist_ok=True)

data_dictionary_path = REPORT_TABLES_DIR / "data_dictionary_initial.csv"
audit_summary_path = REPORT_TABLES_DIR / "data_audit_controls.csv"
missing_summary_path = REPORT_TABLES_DIR / "missing_values_summary.csv"

data_dictionary_initial.to_csv(data_dictionary_path, index=False)
audit_control_summary.to_csv(audit_summary_path, index=False)
missing_summary.to_csv(missing_summary_path, index=False)

print("Saved:")
print(f"- {data_dictionary_path.relative_to(PROJECT_ROOT)}")
print(f"- {audit_summary_path.relative_to(PROJECT_ROOT)}")
print(f"- {missing_summary_path.relative_to(PROJECT_ROOT)}")

Saved:
- reports\tables\data_dictionary_initial.csv
- reports\tables\data_audit_controls.csv
- reports\tables\missing_values_summary.csv


## 20. Stop and review before continuing

Please share the following outputs before any cleaning or EDA decisions are made:

1. `dataset_overview`
2. `dtype_summary`
3. Non-zero rows from `missing_summary`
4. `exact_duplicate_summary`
5. `key_summary`
6. `date_audit`
7. The printed train/test column differences
8. `common_dtype_comparison`
9. `business_rule_summary`
10. `referential_integrity_summary`
11. `submission_alignment`
12. Controls shown under **Controls requiring review**

Also report any warning or error raised while running the notebook.

### Questions to answer together

- Is the confirmed grain one row per store and date?
- What is the usable historical date range?
- What forecast horizon does the Kaggle test represent?
- Which missing values are genuine quality issues and which are structurally expected?
- Are closed stores represented consistently?
- Which variables are unavailable at forecast creation time?
- Which data-type differences must be standardised before merging train and test?